# LeetCode 2036 — Maximum Alternating Subarray Sum bằng Python

Link đề: https://leetcode.com/problems/maximum-alternating-subarray-sum/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming

> **Bài Premium:** đề trong notebook được viết lại theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, nên đối chiếu lại câu chữ và ví dụ với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def maximumAlternatingSubarraySum(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu **tổng đan dấu** của đoạn con: phần tử đầu đoạn mang dấu `+`, rồi xen kẽ `−, +, −, ...` — dấu phụ thuộc vào **vị trí trong đoạn**, không phải chỉ số toàn cục.
2. Mở rộng **thuật toán Kadane** thành DP **hai trạng thái**: đoạn kết thúc tại `i` mà `nums[i]` mang dấu `+` hoặc dấu `−`.
3. Dùng **tổng tiền tố có dấu** `S` và quy về "hiệu lớn nhất của hai tiền tố cùng/khác tính chẵn lẻ".
4. Tối ưu bộ nhớ về `O(1)`, cẩn thận **thứ tự cập nhật** hai biến.
5. Kiểm thử đối chiếu brute force trên mảng có số âm (đáp án có thể âm) với seed cố định.

## 2. Đề bài & ví dụ

Một **đoạn con** là dãy liên tiếp, không rỗng. **Tổng đan dấu** của đoạn `nums[i..j]` là
`nums[i] − nums[i+1] + nums[i+2] − ... ± nums[j]`.

Trả về tổng đan dấu **lớn nhất** trong mọi đoạn con.

**Ví dụ 1:** `[3,-1,1,2]` → `5`: đoạn `[3,-1,1]` cho `3 − (−1) + 1 = 5`.

**Ví dụ 2:** `[2,2,2,2,2]` → `2`: các đoạn `[2]`, `[2,2,2]`, `[2,2,2,2,2]` đều cho 2.

**Ví dụ 3:** `[1]` → `1`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `−10⁵ ≤ nums[i] ≤ 10⁵`.

In [1]:
import random
import time
from itertools import accumulate, product
from typing import List

examples = [([3, -1, 1, 2], 5), ([2, 2, 2, 2, 2], 2), ([1], 1)]
def alt_sum(seg):
    return sum(v if k % 2 == 0 else -v for k, v in enumerate(seg))
for nums, exp in examples:
    print(nums, "-> kỳ vọng", exp)
print("alt_sum([3,-1,1]) =", alt_sum([3, -1, 1]))

[3, -1, 1, 2] -> kỳ vọng 5
[2, 2, 2, 2, 2] -> kỳ vọng 2
[1] -> kỳ vọng 1
alt_sum([3,-1,1]) = 5


## 3. Kiến thức nền

### 3.1 Tổng đan dấu và vị trí trong đoạn

Dấu của `nums[k]` trong đoạn bắt đầu tại `i` là `+` nếu `k − i` chẵn, `−` nếu lẻ. Cùng phần tử có thể mang dấu khác nhau ở hai đoạn khác nhau.

In [2]:
nums = [3, -1, 1, 2]
for i in range(len(nums)):
    signs = ["+" if (k - i) % 2 == 0 else "-" for k in range(i, len(nums))]
    print(f"bắt đầu tại {i}: dấu của nums[{i}..3] = {signs}")

bắt đầu tại 0: dấu của nums[0..3] = ['+', '-', '+', '-']
bắt đầu tại 1: dấu của nums[1..3] = ['+', '-', '+']
bắt đầu tại 2: dấu của nums[2..3] = ['+', '-']
bắt đầu tại 3: dấu của nums[3..3] = ['+']


### 3.2 Thuật toán Kadane (tổng đoạn con lớn nhất)

`best_end = max(x, best_end + x)` — đoạn tốt nhất kết thúc tại vị trí hiện tại hoặc bắt đầu mới tại `x`, hoặc nối dài đoạn trước. Đáp án là max mọi `best_end`.

In [3]:
def kadane(a):
    best_end = ans = a[0]
    for x in a[1:]:
        best_end = max(x, best_end + x)
        ans = max(ans, best_end)
    return ans
print(kadane([-2, 1, -3, 4, -1, 2, 1, -5, 4]))

6


### 3.3 Gán song song `a, b = ..., ...`

Python tính **toàn bộ vế phải trước** rồi mới gán ⇒ cập nhật hai trạng thái phụ thuộc lẫn nhau mà không cần biến tạm. Viết hai dòng riêng thì dòng sau sẽ dùng giá trị **mới** của dòng trước.

In [4]:
f, g = 1, 10
f, g = g + 1, f + 1          # dùng f, g CŨ
print(f, g)
f, g = 1, 10
f = g + 1; g = f + 1         # dòng 2 dùng f MỚI
print(f, g)

11 2
11 12


### 3.4 Tổng tiền tố có dấu

Đặt `s[k] = (−1)^k · nums[k]`, `S[0] = 0`, `S[k+1] = S[k] + s[k]`. Khi đó tổng đan dấu của `nums[i..j]` = `(−1)^i · (S[j+1] − S[i])`.

In [5]:
nums = [3, -1, 1, 2]
s = [v if k % 2 == 0 else -v for k, v in enumerate(nums)]
S = [0] + list(accumulate(s))
i, j = 1, 3
print("s =", s, "S =", S)
print("alt_sum(nums[1..3]) =", alt_sum(nums[1:4]), "= (-1)^1 * (S[4] - S[1]) =", -(S[4] - S[1]))

s = [3, 1, 1, -2] S = [0, 3, 4, 5, 3]
alt_sum(nums[1..3]) = 0 = (-1)^1 * (S[4] - S[1]) = 0


### 3.5 Âm vô cùng `float('-inf')` làm "trạng thái chưa tồn tại"

`−∞ + x = −∞`, `max(−∞, x) = x`, `−∞ < mọi số`. Dùng `−∞` cho trạng thái không thể xảy ra (ví dụ "đoạn kết thúc tại 0 mà phần tử cuối mang dấu −") giúp công thức DP chạy đồng nhất, không cần `if` đặc biệt. Chỉ cần cẩn thận: `−∞ − (−∞)` là `nan` — ở bài này không bao giờ xảy ra vì `f` luôn hữu hạn.

In [6]:
NEG_demo = float("-inf")
print(NEG_demo + 5, max(NEG_demo, -10**9), NEG_demo < -10**18, NEG_demo - NEG_demo)

-inf -1000000000 True nan


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `[3, -1, 1, 2]` — liệt kê tay một số đoạn:

| Đoạn | Tổng đan dấu |
|---|---|
| [3] | 3 |
| [3, −1] | 3 + 1 = 4 |
| [3, −1, 1] | 4 + 1 = **5** |
| [3, −1, 1, 2] | 5 − 2 = 3 |
| [−1, 1, 2] | −1 − 1 + 2 = 0 |
| [1, 2] | −1 |
| [2] | 2 |

Nhận xét: khi kéo dài đoạn sang phải thêm một phần tử, phần tử mới mang dấu **ngược** với phần tử cuối cũ. Vậy nếu biết "đoạn tốt nhất kết thúc tại `i−1` với `nums[i−1]` mang dấu `+`" và "... mang dấu `−`", ta suy ra được cho `i` — đó là DP hai trạng thái.

In [7]:
nums = [3, -1, 1, 2]; n = len(nums)
for i in range(n):
    print(f"bắt đầu {i}:", [alt_sum(nums[i:j + 1]) for j in range(i, n)])

bắt đầu 0: [3, 4, 5, 3]
bắt đầu 1: [-1, -2, 0]
bắt đầu 2: [1, -1]
bắt đầu 3: [2]


## 5. Cách 1 — Brute Force / baseline: mọi đoạn, cộng dồn

**Ý tưởng:** cố định đầu `i`, kéo dài `j`, cộng `±nums[j]` theo `(j − i)` chẵn/lẻ.

In [8]:
def alt_brute(nums):
    n = len(nums); best = None
    for i in range(n):
        cur = 0
        for j in range(i, n):
            cur += nums[j] if (j - i) % 2 == 0 else -nums[j]
            if best is None or cur > best:
                best = cur
    return best

for nums, exp in examples:
    got = alt_brute(nums); print(got, "kỳ vọng", exp); assert got == exp

5 kỳ vọng 5
2 kỳ vọng 2
1 kỳ vọng 1


**Độ phức tạp:** `O(n²)` — `n = 10⁵` ⇒ `5·10⁹` bước, TLE. Space `O(1)`.

**Nút thắt:** đoạn tốt nhất kết thúc tại `j` có thể suy từ thông tin tại `j − 1`, không cần thử mọi điểm đầu.

In [9]:
rng = random.Random(0)
for n in (1000, 2000, 4000):
    a = [rng.randint(-10**5, 10**5) for _ in range(n)]
    t = time.perf_counter(); alt_brute(a)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=1000: 0.031s
n=2000: 0.119s


n=4000: 0.487s


## 6. Ý tưởng thuật toán / bước đột phá

**Hai trạng thái tại vị trí `i`:**
- `f[i]` = tổng đan dấu lớn nhất của đoạn **kết thúc tại `i`** mà `nums[i]` mang dấu **`+`**.
- `g[i]` = ... mà `nums[i]` mang dấu **`−`** (đoạn có ít nhất 2 phần tử).

**Chuyển trạng thái:**
- `nums[i]` mang `+`: hoặc đoạn **bắt đầu** tại `i` (giá trị `nums[i]`), hoặc nối sau một đoạn mà `nums[i−1]` mang `−`: `f[i] = max(nums[i], g[i−1] + nums[i])`.
- `nums[i]` mang `−`: **không thể** bắt đầu tại `i` (phần tử đầu luôn `+`), nên phải nối sau đoạn mà `nums[i−1]` mang `+`: `g[i] = f[i−1] − nums[i]`.
- Khởi tạo: `f[0] = nums[0]`, `g[0] = −∞`.

Đáp án `= max_i max(f[i], g[i])`.

**Quy tắc "nếu… thì…":** **nếu** `g[i−1] < 0` (nối vào chỉ làm giảm) **thì** bắt đầu đoạn mới tại `i` — giống Kadane.

**Góc nhìn tiền tố:** với `S` có dấu, đoạn bắt đầu tại `i` **chẵn** có tổng `S[j+1] − S[i]`; bắt đầu tại `i` **lẻ** có tổng `S[i] − S[j+1]`. Ta duyệt `j` và giữ `min S[i]` với `i` chẵn, `max S[i]` với `i` lẻ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP hai mảng `f`, `g`

**Giải thích từng đoạn:**
1. `f = [0]*n`, `g = [0]*n`; `f[0] = nums[0]`, `g[0] = −∞`.
2. Với `i ≥ 1`: `f[i] = max(nums[i], g[i−1] + nums[i])`, `g[i] = f[i−1] − nums[i]`.
3. Trả `max(max(f), max(g))`.

In [10]:
NEG = float("-inf")

def alt_dp_arrays(nums):
    n = len(nums)
    f = [NEG] * n; g = [NEG] * n
    f[0] = nums[0]
    for i in range(1, n):
        f[i] = max(nums[i], g[i - 1] + nums[i])
        g[i] = f[i - 1] - nums[i]
    return max(max(f), max(g))

for nums, exp in examples:
    assert alt_dp_arrays(nums) == exp
print([alt_dp_arrays(nums) for nums, _ in examples])

[5, 2, 1]


**Độ phức tạp Cách 2:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — DP hai biến `O(1)` bộ nhớ (bản nộp chính)

Chỉ cần `f[i−1]`, `g[i−1]` ⇒ hai biến, cập nhật **song song**.

In [11]:
def alt_dp(nums):
    f, g = nums[0], NEG
    ans = f
    for x in nums[1:]:
        f, g = max(x, g + x), f - x          # vế phải dùng f, g CŨ
        ans = max(ans, f, g)
    return ans

for nums, exp in examples:
    assert alt_dp(nums) == exp
print([alt_dp(nums) for nums, _ in examples])

[5, 2, 1]


In [12]:
class SolutionDP:
    def maximumAlternatingSubarraySum(self, nums: List[int]) -> int:
        f, g = nums[0], float("-inf")
        ans = f
        for x in nums[1:]:
            f, g = max(x, g + x), f - x
            ans = max(ans, f, g)
        return ans

print([SolutionDP().maximumAlternatingSubarraySum(nums) for nums, _ in examples])

[5, 2, 1]


**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(1)` bộ nhớ. Ghi chú: `−∞ + x = −∞` trong Python với `float`, nên không cần xử lý riêng `g` lúc đầu (cuối cùng `ans` luôn là số nguyên vì `f` hữu hạn ở mọi bước).

### 7.3 Cách 4 — Tổng tiền tố có dấu + min/max theo tính chẵn lẻ

**Giải thích từng đoạn:**
1. `S` tiền tố của `s[k] = (−1)^k nums[k]`.
2. Duyệt `j` từ `0..n−1` (đoạn kết thúc tại `j`, vế phải là `S[j+1]`). Trước đó cập nhật ứng viên đầu đoạn `i = j`: nếu `j` chẵn thì `min_even = min(min_even, S[j])`, nếu lẻ thì `max_odd = max(max_odd, S[j])`.
3. Ứng viên: `S[j+1] − min_even` (đầu chẵn) và `max_odd − S[j+1]` (đầu lẻ).

In [13]:
def alt_prefix(nums):
    S = 0                         # S hiện tại = S[j]
    min_even, max_odd = float("inf"), NEG
    ans = NEG
    for j, x in enumerate(nums):
        if j % 2 == 0:
            min_even = min(min_even, S)      # i = j là điểm đầu chẵn
        else:
            max_odd = max(max_odd, S)        # i = j là điểm đầu lẻ
        S += x if j % 2 == 0 else -x         # S = S[j+1]
        ans = max(ans, S - min_even, max_odd - S)
    return ans

for nums, exp in examples:
    assert alt_prefix(nums) == exp
print([alt_prefix(nums) for nums, _ in examples])

[5, 2, 1]


**Độ phức tạp Cách 4:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.4 Cách 5 — Kadane trên hai mảng đổi dấu, chỉ "khởi động lại" đúng tính chẵn lẻ

Đoạn bắt đầu tại `i` chẵn có tổng = tổng đoạn của `s` (dấu toàn cục `(−1)^k`); bắt đầu tại `i` lẻ có tổng = tổng đoạn của `−s`. Chạy Kadane trên `s` nhưng chỉ được **bắt đầu mới** tại chỉ số chẵn, và trên `−s` chỉ bắt đầu mới tại chỉ số lẻ.

In [14]:
def alt_two_kadane(nums):
    s = [v if k % 2 == 0 else -v for k, v in enumerate(nums)]
    def kadane_restart(a, parity):
        best_end, ans = NEG, NEG
        for k, x in enumerate(a):
            if k % 2 == parity:
                best_end = max(x, best_end + x)   # được phép bắt đầu mới tại k
            else:
                best_end = best_end + x           # chỉ được nối dài
            ans = max(ans, best_end)
        return ans
    return max(kadane_restart(s, 0), kadane_restart([-v for v in s], 1))

for nums, exp in examples:
    assert alt_two_kadane(nums) == exp
print([alt_two_kadane(nums) for nums, _ in examples])

[5, 2, 1]


**Độ phức tạp Cách 5:** `O(n)` thời gian, `O(n)` bộ nhớ (mảng `s`; có thể bỏ bằng cách tính dấu tại chỗ).

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 `[3, -1, 1, 2]`:

| i | x | f mới = max(x, g + x) | g mới = f − x | ans |
|---|---|---|---|---|
| 0 | 3 | 3 | −∞ | 3 |
| 1 | −1 | max(−1, −∞) = −1 | 3 − (−1) = 4 | 4 |
| 2 | 1 | max(1, 4 + 1) = 5 | −1 − 1 = −2 | **5** |
| 3 | 2 | max(2, −2 + 2) = 2 | 5 − 2 = 3 | 5 |

In [15]:
def trace_dp(nums):
    f, g = nums[0], NEG; ans = f
    print(f"i=0 x={nums[0]}: f={f}, g={g}, ans={ans}")
    for i, x in enumerate(nums[1:], 1):
        nf, ng = max(x, g + x), f - x
        print(f"i={i} x={x}: f = max({x}, {g} + {x}) = {nf}; g = {f} - {x} = {ng}", end="")
        f, g = nf, ng; ans = max(ans, f, g)
        print(f" -> ans={ans}")
    return ans

print("Đáp án:", trace_dp([3, -1, 1, 2]))
print()
print("Đáp án:", trace_dp([2, 2, 2, 2, 2]))

i=0 x=3: f=3, g=-inf, ans=3
i=1 x=-1: f = max(-1, -inf + -1) = -1; g = 3 - -1 = 4 -> ans=4
i=2 x=1: f = max(1, 4 + 1) = 5; g = -1 - 1 = -2 -> ans=5
i=3 x=2: f = max(2, -2 + 2) = 2; g = 5 - 2 = 3 -> ans=5
Đáp án: 5

i=0 x=2: f=2, g=-inf, ans=2
i=1 x=2: f = max(2, -inf + 2) = 2; g = 2 - 2 = 0 -> ans=2
i=2 x=2: f = max(2, 0 + 2) = 2; g = 2 - 2 = 0 -> ans=2
i=3 x=2: f = max(2, 0 + 2) = 2; g = 2 - 2 = 0 -> ans=2
i=4 x=2: f = max(2, 0 + 2) = 2; g = 2 - 2 = 0 -> ans=2
Đáp án: 2


In [16]:
def trace_prefix(nums):
    S = 0; min_even, max_odd = float("inf"), NEG; ans = NEG
    for j, x in enumerate(nums):
        if j % 2 == 0: min_even = min(min_even, S)
        else: max_odd = max(max_odd, S)
        S += x if j % 2 == 0 else -x
        c1, c2 = S - min_even, max_odd - S
        ans = max(ans, c1, c2)
        print(f"j={j}: S[j+1]={S:3}, min_even={min_even}, max_odd={max_odd}, đầu chẵn={c1}, đầu lẻ={c2}, ans={ans}")
    return ans

print("Đáp án:", trace_prefix([3, -1, 1, 2]))

j=0: S[j+1]=  3, min_even=0, max_odd=-inf, đầu chẵn=3, đầu lẻ=-inf, ans=3
j=1: S[j+1]=  4, min_even=0, max_odd=3, đầu chẵn=4, đầu lẻ=-1, ans=4
j=2: S[j+1]=  5, min_even=0, max_odd=3, đầu chẵn=5, đầu lẻ=-2, ans=5
j=3: S[j+1]=  3, min_even=0, max_odd=5, đầu chẵn=3, đầu lẻ=2, ans=5
Đáp án: 5


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def maximumAlternatingSubarraySum(self, nums: List[int]) -> int:
        # f: đoạn tốt nhất kết thúc tại i, nums[i] mang dấu +
        # g: đoạn tốt nhất kết thúc tại i, nums[i] mang dấu -
        f, g = nums[0], float("-inf")
        ans = f
        for x in nums[1:]:
            f, g = max(x, g + x), f - x     # cập nhật song song bằng giá trị cũ
            if f > ans: ans = f
            if g > ans: ans = g
        return ans

for nums, exp in examples:
    assert Solution().maximumAlternatingSubarraySum(nums) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Định nghĩa.** `F(i)` = max tổng đan dấu của đoạn `[l..i]` có `(i − l)` chẵn (`nums[i]` mang `+`); `G(i)` = max với `(i − l)` lẻ (`nums[i]` mang `−`), `G(i) = −∞` nếu không có đoạn như vậy.

**Quy nạp.** `F(0) = nums[0]`, `G(0) = −∞` (đoạn duy nhất kết thúc tại 0 là `[0..0]`, dấu `+`).

Với `i ≥ 1`, một đoạn `[l..i]` có `nums[i]` mang `+`:
- hoặc `l = i` — giá trị `nums[i]`;
- hoặc `l < i` — khi đó `[l..i−1]` là đoạn kết thúc tại `i − 1` với `nums[i−1]` mang `−`, và tổng = `alt(l..i−1) + nums[i]`; tối đa là `G(i−1) + nums[i]`.

⇒ `F(i) = max(nums[i], G(i−1) + nums[i])`. Tương tự, đoạn có `nums[i]` mang `−` buộc `l < i` và `[l..i−1]` có `nums[i−1]` mang `+` ⇒ `G(i) = F(i−1) − nums[i]`.

Mọi đoạn con kết thúc tại một `i` nào đó với một trong hai dấu ⇒ đáp án `= max_i max(F(i), G(i))`. Cách 3 giữ đúng `F(i−1), G(i−1)` nhờ gán song song.

Kiểm chứng **từng trạng thái** (không chỉ đáp án cuối): với mỗi `i`, so `f[i]` của Cách 2 với max tổng đan dấu của mọi đoạn `[l..i]` có `(i − l)` chẵn, và `g[i]` với các đoạn có `(i − l)` lẻ (tính bằng brute force).

In [18]:
def dp_states(nums):
    n = len(nums); f = [NEG] * n; g = [NEG] * n; f[0] = nums[0]
    for i in range(1, n):
        f[i] = max(nums[i], g[i - 1] + nums[i]); g[i] = f[i - 1] - nums[i]
    return f, g

rng = random.Random(10)
for _ in range(300):
    a = [rng.randint(-9, 9) for _ in range(rng.randint(1, 12))]
    f, g = dp_states(a)
    for i in range(len(a)):
        F = max(alt_sum(a[l:i + 1]) for l in range(0, i + 1) if (i - l) % 2 == 0)
        G = max((alt_sum(a[l:i + 1]) for l in range(0, i + 1) if (i - l) % 2 == 1), default=NEG)
        assert f[i] == F and g[i] == G
f, g = dp_states([3, -1, 1, 2])
print("F(i), G(i) đúng với định nghĩa trên 300 mảng; ví dụ 1: f =", f, "g =", g)

F(i), G(i) đúng với định nghĩa trên 300 mảng; ví dụ 1: f = [3, -1, 5, 2] g = [-inf, 4, -2, 3]


## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` |
| 2. DP hai mảng | `O(n)` | `O(n)` |
| 3. DP hai biến | `O(n)` | `O(1)` |
| 4. Tiền tố có dấu | `O(n)` | `O(1)` |
| 5. Hai Kadane | `O(n)` | `O(n)` |

Giá trị tuyệt đối đáp án ≤ `n · 10⁵ = 10¹⁰` — vượt `int32`, Python không lo, Java/C++ dùng `long`.

In [19]:
rng = random.Random(2036)
big = [rng.randint(-10**5, 10**5) for _ in range(10**5)]
worst = [10**5 if k % 2 == 0 else -10**5 for k in range(10**5)]      # đáp án 10^10
for name, fn in [("dp_arrays", alt_dp_arrays), ("dp", alt_dp), ("prefix", alt_prefix), ("two_kadane", alt_two_kadane),
                 ("Solution", lambda a: Solution().maximumAlternatingSubarraySum(a))]:
    t = time.perf_counter(); r1 = fn(big); r2 = fn(worst)
    print(f"{name:10s}: ngẫu nhiên={r1}, xen kẽ ±10^5={r2} ({time.perf_counter() - t:.3f}s)")

dp_arrays : ngẫu nhiên=24224372, xen kẽ ±10^5=10000000000 (0.026s)
dp        : ngẫu nhiên=24224372, xen kẽ ±10^5=10000000000 (0.014s)
prefix    : ngẫu nhiên=24224372, xen kẽ ±10^5=10000000000 (0.025s)
two_kadane: ngẫu nhiên=24224372, xen kẽ ±10^5=10000000000 (0.039s)
Solution  : ngẫu nhiên=24224372, xen kẽ ±10^5=10000000000 (0.012s)


## 12. Bộ test

- Ví dụ đề.
- Biên: một phần tử âm, toàn số âm (đáp án là phần tử lớn nhất hoặc một hiệu dương!), toàn số 0, hai phần tử.
- **Vét cạn** mọi mảng độ dài `1..6` trên `{−2, −1, 0, 1, 2}`, so với brute force.
- Ngẫu nhiên dài hơn với giá trị lớn (seed cố định).

In [20]:
fns = {
    "brute": alt_brute, "dp_arrays": alt_dp_arrays, "dp": alt_dp, "prefix": alt_prefix, "two_kadane": alt_two_kadane,
    "Solution": lambda a: Solution().maximumAlternatingSubarraySum(a),
    "SolutionDP": lambda a: SolutionDP().maximumAlternatingSubarraySum(a),
}
edge = [([-5], -5), ([-3, -7], 4), ([0, 0, 0], 0), ([-1, -1, -1], 0), ([5, 10], 10), ([1, -2, 3, -4], 10)]
for nums, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(nums) == exp, (name, nums, fn(nums))
cnt = 0
for n in range(1, 7):
    for tup in product((-2, -1, 0, 1, 2), repeat=n):
        a = list(tup); want = alt_brute(a)
        for name, fn in fns.items():
            assert fn(a) == want, (name, a)
        cnt += 1
print(f"Ví dụ + biên + vét cạn {cnt} mảng: OK")

Ví dụ + biên + vét cạn 19530 mảng: OK


In [21]:
rng = random.Random(42)
for _ in range(300):
    a = [rng.randint(-10**5, 10**5) for _ in range(rng.randint(1, 200))]
    want = alt_brute(a)
    for name, fn in fns.items():
        assert fn(a) == want, (name, a)
print("300 test ngẫu nhiên (giá trị lớn): OK")

300 test ngẫu nhiên (giá trị lớn): OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng dấu theo **chỉ số toàn cục** (chỉ xét đoạn bắt đầu tại chỉ số chẵn)

Lấy `s[k] = (−1)^k·nums[k]` rồi chạy Kadane thường là sai: đoạn bắt đầu ở chỉ số **lẻ** cần dấu ngược lại, và Kadane thường còn cho phép bắt đầu ở chỉ số lẻ với dấu sai.

In [22]:
def wrong_global_sign(nums):
    s = [v if k % 2 == 0 else -v for k, v in enumerate(nums)]
    return kadane(s)                                   # SAI

a = [-5, 3]
print("sai:", wrong_global_sign(a), "| đúng:", alt_brute(a))

sai: -3 | đúng: 3


In [23]:
print("sửa (DP hai trạng thái):", alt_dp(a))

sửa (DP hai trạng thái): 3


### Lỗi 2 — Cho phép đoạn **bắt đầu bằng dấu −** (`g[i] = max(−x, f + ... )`)

Phần tử đầu đoạn luôn mang dấu `+`. Nếu cho `g` "bắt đầu mới" với `−x`, với `[-10]` ta nhận `10` thay vì `−10`.

In [24]:
def wrong_start_minus(nums):
    f, g = nums[0], -nums[0]                            # SAI: g bắt đầu từ -nums[0]
    ans = max(f, g)
    for x in nums[1:]:
        f, g = max(x, g + x), max(-x, f - x)           # SAI: max(-x, ...)
        ans = max(ans, f, g)
    return ans

for a in ([-10], [3, -1, 1, 2]):
    print(a, "sai:", wrong_start_minus(a), "| đúng:", alt_brute(a))

[-10] sai: 10 | đúng: -10
[3, -1, 1, 2] sai: 5 | đúng: 5


In [25]:
print("sửa:", alt_dp([-10]), alt_dp([3, -1, 1, 2]))

sửa: -10 5


### Lỗi 3 — Khởi tạo `ans = 0`

Đoạn phải **không rỗng**; với mảng `[-5]` đáp án là `−5`, không phải `0`.

In [26]:
def wrong_init_zero(nums):
    f, g, ans = nums[0], NEG, 0                         # SAI: ans = 0
    ans = max(ans, f)
    for x in nums[1:]:
        f, g = max(x, g + x), f - x
        ans = max(ans, f, g)
    return ans

print("sai:", wrong_init_zero([-5]), "| đúng:", alt_brute([-5]))

sai: 0 | đúng: -5


In [27]:
print("sửa (ans = nums[0]):", alt_dp([-5]))

sửa (ans = nums[0]): -5


### Lỗi 4 — Cập nhật tuần tự (dùng `f` **mới** để tính `g`)

`f = max(x, g + x)` rồi `g = f - x` — dòng thứ hai dùng `f` vừa cập nhật (đoạn kết thúc tại `i`), không phải `f` của `i − 1`.

In [28]:
def wrong_sequential(nums):
    f, g = nums[0], NEG; ans = f
    for x in nums[1:]:
        f = max(x, g + x)
        g = f - x                                       # SAI: f đã là f mới
        ans = max(ans, f, g)
    return ans

print("sai:", wrong_sequential([3, -1, 1, 2]), "| đúng:", alt_brute([3, -1, 1, 2]))

sai: 3 | đúng: 5


In [29]:
print("sửa (gán song song):", alt_dp([3, -1, 1, 2]))

sửa (gán song song): 5


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đoạn con tốt nhất" với dấu/trạng thái thay đổi theo vị trí trong đoạn** ⇒ Kadane **nhiều trạng thái** (ở đây 2: dấu của phần tử cuối).
- Trạng thái nào **không thể bắt đầu** đoạn (dấu `−`) thì chỉ được chuyển từ trạng thái kia — khởi tạo `−∞`.
- Hai trạng thái phụ thuộc chéo ⇒ **gán song song** `f, g = ..., ...`.
- Tổng đan dấu = tổng thường trên mảng đổi dấu toàn cục, **nhân** `(−1)^i` của điểm đầu ⇒ góc nhìn tiền tố.
- Mảng có số âm ⇒ đáp án có thể âm ⇒ khởi tạo bằng phần tử đầu / `−∞`, không bằng 0.

## 15. Biến thể của bài

1. **LeetCode 1911 — Maximum Alternating Subsequence Sum:** **dãy con** (không cần liên tiếp) — DP hai trạng thái "chọn/không chọn" (bài tập 5).
2. **LeetCode 53 — Maximum Subarray:** Kadane một trạng thái.
3. **LeetCode 152 — Maximum Product Subarray:** Kadane hai trạng thái (max và min vì số âm đổi dấu).
4. **Tổng đan dấu nhỏ nhất:** đổi `max` thành `min`, hoặc áp dụng bài gốc cho `−nums`? (Cẩn thận: dấu của phần tử đầu vẫn là `+`, nên `min alt(nums) = −max alt(−nums)`.)
5. **Tổng đan dấu lớn nhất với độ dài chẵn / lẻ cố định** — chỉ lấy `ans` từ `f` (độ dài lẻ) hoặc `g` (độ dài chẵn).

Minh họa biến thể 4 và 5 (tự dựng, kiểm bằng brute force):

In [30]:
def alt_min(nums):
    return -alt_dp([-v for v in nums])

def alt_max_by_parity(nums):
    f, g = nums[0], NEG; best_odd, best_even = f, NEG
    for x in nums[1:]:
        f, g = max(x, g + x), f - x
        best_odd, best_even = max(best_odd, f), max(best_even, g)
    return best_odd, best_even

rng = random.Random(7)
for _ in range(400):
    a = [rng.randint(-9, 9) for _ in range(rng.randint(1, 10))]
    n = len(a)
    segs = [(alt_sum(a[i:j + 1]), j - i + 1) for i in range(n) for j in range(i, n)]
    assert alt_min(a) == min(v for v, _ in segs)
    bo, be = alt_max_by_parity(a)
    assert bo == max(v for v, L in segs if L % 2 == 1)
    assert be == max((v for v, L in segs if L % 2 == 0), default=NEG)
print("Biến thể min và theo độ dài chẵn/lẻ khớp brute force; [3,-1,1,2] ->", alt_min([3, -1, 1, 2]), alt_max_by_parity([3, -1, 1, 2]))

Biến thể min và theo độ dài chẵn/lẻ khớp brute force; [3,-1,1,2] -> -2 (5, 4)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | mọi đoạn, cộng dồn | `O(n²)` | `O(1)` | kiểm tra |
| 2. DP hai mảng | `f[i]`, `g[i]` | `O(n)` | `O(n)` | dễ debug, in bảng |
| 3. DP hai biến | gán song song | `O(n)` | `O(1)` | **nộp bài** |
| 4. Tiền tố có dấu | min `S` chẵn / max `S` lẻ | `O(n)` | `O(1)` | khi quen kỹ thuật tiền tố |
| 5. Hai Kadane | khởi động lại theo tính chẵn lẻ | `O(n)` | `O(n)` | tái sử dụng Kadane |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `[1, -2, 3, -4]`.
2. **Tính tay:** bảng `f, g, ans` cho `[-3, -7, 2]`.
3. **Điền TODO:** `best_segment(nums)` trả về `(l, r)` của một đoạn đạt tổng đan dấu lớn nhất.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `max_alt_subsequence(nums)` — tổng đan dấu lớn nhất của một **dãy con** (không cần liên tiếp, không rỗng, chỉ số tăng dần; phần tử đầu dấu `+`). Kiểm bằng brute force mọi tập con (`n ≤ 10`).

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_segment(nums):
    # TODO: như Cách 3, nhưng lưu điểm bắt đầu của f và g
    pass

In [32]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_max_alt(nums):
    # TODO
    pass

In [33]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def max_alt_subsequence(nums):
    # TODO: even = tốt nhất khi phần tử cuối dấu +, odd = khi phần tử cuối dấu -
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Cả đoạn: `1 + 2 + 3 + 4 = 10`. (2) Không đoạn nào lớn hơn. (3) `10`.

**Bài 2.** (1) `i=0`: `f=−3, g=−∞`. (2) `i=1`: `f = max(−7, −∞) = −7`, `g = −3 − (−7) = 4`. (3) `i=2`: `f = max(2, 4 + 2) = 6`, `g = −7 − 2 = −9` ⇒ đáp án `6`.

**Bài 3.** (1) Lưu `fl`, `gl` = điểm đầu của đoạn tương ứng. (2) Khi `f` chọn "bắt đầu mới" thì `fl = i`; khi nối từ `g` thì `fl = gl` cũ. (3) `g` luôn nối từ `f` cũ ⇒ `gl = fl` cũ.

**Bài 4.** (1) `f = nums[0], g = −∞`. (2) Gán song song. (3) `ans` khởi tạo `nums[0]`.

**Bài 5.** (1) `even` = tốt nhất với phần tử cuối mang `+`, `odd` = mang `−`. (2) Với mỗi `x`: `even = max(even, odd + x, x)`, `odd = max(odd, even_cũ − x)`. (3) Đáp án `max(even, odd)` (có thể bỏ `odd` nếu luôn kết thúc bằng `+` tốt hơn).

## 19. Đáp án tham khảo

In [34]:
# Bài 1, 2
assert Solution().maximumAlternatingSubarraySum([1, -2, 3, -4]) == 10
assert Solution().maximumAlternatingSubarraySum([-3, -7, 2]) == 6
print("Bài 1: 10 | Bài 2: 6")
trace_dp([-3, -7, 2])

Bài 1: 10 | Bài 2: 6
i=0 x=-3: f=-3, g=-inf, ans=-3
i=1 x=-7: f = max(-7, -inf + -7) = -7; g = -3 - -7 = 4 -> ans=4
i=2 x=2: f = max(2, 4 + 2) = 6; g = -7 - 2 = -9 -> ans=6


6

In [35]:
# Bài 3, 4
def best_segment_ans(nums):
    f, fl = nums[0], 0; g, gl = NEG, -1
    best, seg = f, (0, 0)
    for i in range(1, len(nums)):
        x = nums[i]
        if x >= g + x: nf, nfl = x, i
        else: nf, nfl = g + x, gl
        ng, ngl = f - x, fl
        f, fl, g, gl = nf, nfl, ng, ngl
        if f > best: best, seg = f, (fl, i)
        if g > best: best, seg = g, (gl, i)
    return seg

my_max_alt_ans = alt_dp
rng = random.Random(3)
for _ in range(400):
    a = [rng.randint(-9, 9) for _ in range(rng.randint(1, 15))]
    l, r = best_segment_ans(a)
    assert alt_sum(a[l:r + 1]) == alt_brute(a) == my_max_alt_ans(a)
print("Bài 3, 4: OK; ví dụ 1 ->", best_segment_ans([3, -1, 1, 2]))

Bài 3, 4: OK; ví dụ 1 -> (0, 2)


In [36]:
# Bài 5
def max_alt_subsequence_ans(nums):
    even, odd = NEG, NEG                  # phần tử cuối mang +, mang -
    for x in nums:
        even, odd = max(even, odd + x, x), max(odd, even - x)
    return max(even, odd)

def max_alt_subsequence_brute(nums):
    n = len(nums); best = NEG
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        best = max(best, alt_sum(seq))
    return best

rng = random.Random(1911)
for _ in range(300):
    a = [rng.randint(-9, 9) for _ in range(rng.randint(1, 10))]
    assert max_alt_subsequence_ans(a) == max_alt_subsequence_brute(a)
print("Bài 5: OK; [4,2,5,3] ->", max_alt_subsequence_ans([4, 2, 5, 3]), "(LeetCode 1911 cho 7)")

Bài 5: OK; [4,2,5,3] -> 7 (LeetCode 1911 cho 7)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template Kadane hai trạng thái:**

```python
f, g = nums[0], float('-inf')     # f: cuối mang +, g: cuối mang -
ans = f
for x in nums[1:]:
    f, g = max(x, g + x), f - x   # song song!
    ans = max(ans, f, g)
return ans
```

In [37]:
def template(nums):
    f, g = nums[0], float("-inf"); ans = f
    for x in nums[1:]:
        f, g = max(x, g + x), f - x
        ans = max(ans, f, g)
    return ans
print([template(a) for a, _ in examples])

[5, 2, 1]


**Checklist:**

- [ ] Dấu theo **vị trí trong đoạn** (phần tử đầu `+`), không theo chỉ số toàn cục.
- [ ] Trạng thái "cuối mang `−`" không được bắt đầu đoạn mới (khởi tạo `−∞`).
- [ ] Gán **song song** `f, g = ..., ...`.
- [ ] `ans` khởi tạo `nums[0]` (đoạn không rỗng, đáp án có thể âm).
- [ ] Lấy max của **cả** `f` và `g`.
- [ ] Đã thử: một phần tử âm, toàn âm, ví dụ đề.